In [42]:
from pyspark.sql import SparkSession

DEFAULT_SPARK_REMOTE = "sc://localhost:15002"
spark = SparkSession.builder.remote(DEFAULT_SPARK_REMOTE).getOrCreate()

spark.version

'4.1.2'

In [43]:
spark.sql("use spark_catalog")
spark.sql("use schema northwind_db")

DataFrame[]

In [44]:
df = spark.table("orders")
df.createOrReplaceTempView("orders_view")

In [45]:
print(f"rows count: {df.count()}")
#print(f"rows distinct count: {df.select("CustomerID").distinct().count()}")

rows count: 830


## count, countDistinct, approx_count_distinct

In [46]:
from pyspark.sql.functions import lit, count, countDistinct, approx_count_distinct

df_agg = (df.select(count("*"), countDistinct("CustomerID"), approx_count_distinct("CustomerID")))
display(df_agg)

DataFrame[count(1): bigint, count(DISTINCT CustomerID): bigint, approx_count_distinct(CustomerID): bigint]

In [47]:
spark.sql("""
select 
    count(*),
    count(distinct CustomerID),
    approx_count_distinct(CustomerID)
from orders_view
""").show()

+--------+--------------------------+---------------------------------+
|count(1)|count(DISTINCT CustomerID)|approx_count_distinct(CustomerID)|
+--------+--------------------------+---------------------------------+
|     830|                        89|                               91|
+--------+--------------------------+---------------------------------+



## first, last, min, max, sum, sum_distinct

In [48]:

from pyspark.sql.functions import first, last

df_agg = df.orderBy("OrderDate").select(first("CustomerID"), last("CustomerID"))
df_agg.show()

+-----------------+----------------+
|first(CustomerID)|last(CustomerID)|
+-----------------+----------------+
|            VINET|           SIMOB|
+-----------------+----------------+



In [49]:
spark.sql("""
select 
    first(CustomerID),
    last(CustomerID)
from orders_view
""").show()

+-----------------+----------------+
|first(CustomerID)|last(CustomerID)|
+-----------------+----------------+
|            BLONP|           RANCH|
+-----------------+----------------+



In [50]:

from pyspark.sql.functions import min, max, avg, sum, col, sum_distinct, expr

df_od = spark.table("order_details")

df_agg = df_od.select(
                      min("Discount"), 
                      max("Discount"),
                      avg("Discount"),
                      expr("mean(Discount)"),
                      sum(col("UnitPrice") * col("Quantity")),
                      sum_distinct("Quantity")
                      )
display(df_agg)

DataFrame[min(Discount): decimal(10,2), max(Discount): decimal(10,2), avg(Discount): decimal(14,6), mean(Discount): decimal(14,6), sum((UnitPrice * Quantity)): decimal(31,2), sum(DISTINCT Quantity): bigint]

In [51]:
spark.sql("""
select 
    min(Discount), 
    max(Discount),
    avg(Discount),
    mean(Discount),
    sum(UnitPrice * Quantity),
    sum(distinct Quantity)
from order_details
""").show()


+-------------+-------------+-------------+--------------+---------------------------+----------------------+
|min(Discount)|max(Discount)|avg(Discount)|mean(Discount)|sum((UnitPrice * Quantity))|sum(DISTINCT Quantity)|
+-------------+-------------+-------------+--------------+---------------------------+----------------------+
|         0.00|         0.25|     0.056167|      0.056167|                 1354458.59|                  2267|
+-------------+-------------+-------------+--------------+---------------------------+----------------------+

